In [1]:
import os, re
import numpy as np
import scanpy as sc
from os.path import join
import pandas as pd

import sys
import scipy.io as sio
import scipy.sparse as sps
import matplotlib.pyplot as plt

from spamosaic.framework import SpaMosaic
import spamosaic.utils as utls
from spamosaic.preprocessing import RNA_preprocess, ADT_preprocess, Epigenome_preprocess, harmony

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8' 

In [13]:
def subset_ad(ad, subset_index):
    ad = ad[subset_index].copy()
    return ad

def reorder(ad1, ad2):
    shared_barcodes = ad1.obs_names.intersection(ad2.obs_names)
    ad1 = ad1[shared_barcodes].copy()
    ad2 = ad2[shared_barcodes].copy()
    return ad1, ad2

import h5py
import gc
def load_h5(path):
    with h5py.File(path, 'r') as f:
        print(f['matrix'].keys())
        print(f['matrix']['features'].keys())

        barcodes = [_.decode('utf-8') for _ in f['matrix']['barcodes'][:]]
        data = f['matrix']['data'][:]
        indices = f['matrix']['indices'][:]
        indptr = f['matrix']['indptr'][:]
        shape = f['matrix']['shape'][:]

        feature_type = [_.decode('utf-8') for _ in f['matrix']['features']['feature_type'][:]]
        feature_id   = [_.decode('utf-8') for _ in f['matrix']['features']['id'][:]]
        feature_name = [_.decode('utf-8') for _ in f['matrix']['features']['name'][:]]
        feature_interval = [_.decode('utf-8') for _ in f['matrix']['features']['interval'][:]]


        X = sps.csc_matrix(
            (data, indices, indptr), 
            shape = shape
        ).tocsc().astype(np.float32).T.toarray()

        adata = sc.AnnData(X)
        adata.obs_names = barcodes
        adata.var_names = feature_id
        adata.var['type'] = feature_type
        adata.var['name'] = feature_name
        adata.var['interval'] = feature_interval
    return adata

def load_peak_expr(_dir):
    data = sio.mmread(join(_dir, 'data.mtx'))
    cname = pd.read_csv(join(_dir, 'barcode.csv'), index_col=0).index.to_list()
    feat = pd.read_csv(join(_dir, 'feat.csv'), index_col=0)['x'].to_list()
    ad = sc.AnnData(sps.csr_matrix(data.T))
    ad.obs_names = cname
    ad.var_names = feat
    return ad

def set_spatial(ad):
    ad.obsm['spatial'] = ad.obs[['array_row', 'array_col']].values
    ad.obsm['spatial'] = ad.obsm['spatial'][:, ::-1]
    ad.obsm['spatial'][:, 1] = -1 * ad.obsm['spatial'][:, 1]
    return ad

In [7]:
data_dir = '../../../data/raw/MISAR_seq/'
ad_bridge = load_h5(join(data_dir, 'E15_5-S1_raw_feature_bc_matrix.h5'))
ad_test1 = load_h5(join(data_dir, 'E13_5-S1_raw_feature_bc_matrix.h5'))
ad_test2 = load_h5(join(data_dir, 'E18_5-S1_raw_feature_bc_matrix.h5'))  # inconsistent peak name across batches
peak_mat = sps.csr_matrix(sio.mmread(join(data_dir, 'BaiduDisk/section1/peak_mat.mtx')).T)
peak_spot_name = pd.read_csv(join(data_dir, 'BaiduDisk/section1/peak_spot_names.csv')).x.values

meta = pd.read_csv(join(data_dir, 'BaiduDisk/section1/meta_data.csv'), index_col=0)

ad_bridge.obs_names = [f'E15_5-S1#{_}' for _ in ad_bridge.obs_names]
ad_test1.obs_names = [f'E13_5-S1#{_}' for _ in ad_test1.obs_names]
ad_test2.obs_names = [f'E18_5-S1#{_}' for _ in ad_test2.obs_names]

# split rna and peak
ad15_rna = ad_bridge[:, ad_bridge.var['type'] == 'Gene Expression'].copy()
ad13_rna = ad_test1[:, ad_test1.var['type'] == 'Gene Expression'].copy()
ad18_rna = ad_test2[:, ad_test2.var['type'] == 'Gene Expression'].copy()

ad13_atac = load_peak_expr(join(data_dir, 'S1-E13-E15-18-peak_data/E13'))
ad15_atac = load_peak_expr(join(data_dir, 'S1-E13-E15-18-peak_data/E15'))
ad18_atac = load_peak_expr(join(data_dir, 'S1-E13-E15-18-peak_data/E18'))
ad13_atac.obs_names = [f'E13_5-S1#{_}' for _ in ad13_atac.obs_names]
ad15_atac.obs_names = [f'E15_5-S1#{_}' for _ in ad15_atac.obs_names]
ad18_atac.obs_names = [f'E18_5-S1#{_}' for _ in ad18_atac.obs_names]

ad15_rna = subset_ad(ad15_rna, ad15_rna.obs_names.intersection(meta.index))
ad13_rna = subset_ad(ad13_rna, ad13_rna.obs_names.intersection(meta.index))
ad18_rna = subset_ad(ad18_rna, ad18_rna.obs_names.intersection(meta.index))

ad15_rna.obs = meta.loc[ad15_rna.obs_names].copy()
ad15_atac.obs = meta.loc[ad15_atac.obs_names].copy()
ad13_rna.obs = meta.loc[ad13_rna.obs_names].copy()
ad13_atac.obs = meta.loc[ad13_atac.obs_names].copy()
ad18_rna.obs = meta.loc[ad18_rna.obs_names].copy()
ad18_atac.obs = meta.loc[ad18_atac.obs_names].copy()

ad15_atac = ad15_atac[ad15_rna.obs_names].copy()
ad13_atac = ad13_atac[ad13_rna.obs_names].copy()
ad18_rna  = ad18_rna[ad18_atac.obs_names].copy()  

del peak_mat, ad_bridge, ad_test1, ad_test2
gc.collect()

<KeysViewHDF5 ['barcodes', 'data', 'features', 'indices', 'indptr', 'shape']>
<KeysViewHDF5 ['_all_tag_keys', 'feature_type', 'genome', 'id', 'interval', 'name']>
<KeysViewHDF5 ['barcodes', 'data', 'features', 'indices', 'indptr', 'shape']>
<KeysViewHDF5 ['_all_tag_keys', 'feature_type', 'genome', 'id', 'interval', 'name']>
<KeysViewHDF5 ['barcodes', 'data', 'features', 'indices', 'indptr', 'shape']>
<KeysViewHDF5 ['_all_tag_keys', 'feature_type', 'genome', 'id', 'interval', 'name']>


4061

In [14]:
ad15_rna = set_spatial(ad15_rna);ad15_atac = set_spatial(ad15_atac)
ad13_rna = set_spatial(ad13_rna);ad13_atac = set_spatial(ad13_atac)
ad18_rna = set_spatial(ad18_rna);ad18_atac = set_spatial(ad18_atac)

In [8]:
ad13_rna.obs['src'] = ad13_atac.obs['src'] = ['e13']*ad13_rna.n_obs
ad15_rna.obs['src'] = ad15_atac.obs['src'] = ['e15']*ad15_rna.n_obs
ad18_rna.obs['src'] = ad18_atac.obs['src'] = ['e18']*ad18_rna.n_obs

ad_rna_all = sc.concat([ad13_rna, ad15_rna, ad18_rna])
ad_atac_all = sc.concat([ad13_atac, ad15_atac, ad18_atac])

sc.pp.highly_variable_genes(ad_rna_all, flavor='seurat_v3', n_top_genes=5000, batch_key='src')
hvg_names = ad_rna_all.var.query('highly_variable').index.to_numpy()

sc.pp.highly_variable_genes(ad_atac_all, flavor='seurat_v3', n_top_genes=50000, batch_key='src')
hvp_names = ad_atac_all.var.query('highly_variable').index.to_numpy()

In [9]:
ad13_rna = ad13_rna[:, hvg_names].copy(); ad13_atac = ad13_atac[:, hvp_names].copy()
ad15_rna = ad15_rna[:, hvg_names].copy(); ad15_atac = ad15_atac[:, hvp_names].copy()
ad18_rna = ad18_rna[:, hvg_names].copy(); ad18_atac = ad18_atac[:, hvp_names].copy()

## filter feat names
filtered_atac_feats = [_ for _ in ad13_atac.var_names if _.startswith('chr')]
ad13_atac = ad13_atac[:, filtered_atac_feats].copy()
ad15_atac = ad15_atac[:, filtered_atac_feats].copy()
ad18_atac = ad18_atac[:, filtered_atac_feats].copy()

In [10]:
RNA_ADS = [ad13_rna, ad15_rna, ad18_rna]
ATAC_ADS = [ad13_atac, ad15_atac, ad18_atac]
mod_dict = {'rna': RNA_ADS, 'atac':ATAC_ADS}
n_batches = 3
mod_sets = ['rna', 'atac']

In [12]:
ad13_rna

AnnData object with n_obs × n_vars = 1777 × 5000
    obs: 'Sample', 'TSSEnrichment', 'ReadsInTSS', 'ReadsInPromoter', 'ReadsInBlacklist', 'PromoterRatio', 'PassQC', 'NucleosomeRatio', 'nMultiFrags', 'nMonoFrags', 'nFrags', 'nDiFrags', 'Gex_RiboRatio', 'Gex_nUMI', 'Gex_nGenes', 'Gex_MitoRatio', 'BlacklistRatio', 'array_col', 'array_row', 'ReadsInPeaks', 'FRIP', 'ATAC_Clusters', 'RNA_Clusters', 'Combined_Clusters', 'Combined_Clusters_annotation', 'src'
    var: 'type', 'name', 'interval'
    obsm: 'dimred_bc'

In [15]:
res = []
for i in range(n_batches):  # train test split
    print(f'cv={i+1}, missing=atac')
    mod_BatchDict = {'rna': RNA_ADS, 
                     'atac': [ATAC_ADS[bi] if bi!=i else None for bi in range(n_batches)]}
    input_key = 'dimred_bc'
    batch_key = 'src'
        
    Epigenome_preprocess(mod_BatchDict['atac'], batch_corr=True, n_peak=50000, batch_key=batch_key, key=input_key, return_hvf=False)
    RNA_preprocess(mod_BatchDict['rna'], favor='scanpy', batch_corr=True, n_hvg=5000, batch_key=batch_key, key=input_key)
    
    model = SpaMosaic(
        modBatch_dict=mod_BatchDict, input_key=input_key,
        batch_key=batch_key, 
        intra_knns=10, inter_knn_base=10, 
        w_g=0.8,
        seed=1234, 
        device='cuda:0'
    )
    
    model.train(net='wlgcn', lr=0.01, T=0.01, n_epochs=100)
    _ = model.infer_emb(mod_BatchDict, emb_key='emb', final_latent_key='merged_emb')

    for k in mod_BatchDict.keys():
        for ad in mod_BatchDict[k]:
            if ad is not None:
                ad.layers['counts'] = sps.csr_matrix(ad.X)

    # for knn in [10, 20, 30]:
    imp_dict = model.impute(mod_BatchDict, emb_key='emb', layer_key='counts', imp_knn=10)

    print(f'==> cv {i}, ATAC imputation: ')
    pr_X = imp_dict['atac'][i] 
    ad_pred = sc.AnnData(pr_X, obs=mod_dict['atac'][i].obs.copy(), var=mod_dict['atac'][i].var.copy())

    out_dir = "../../../results/imputations/SpaMosaic/Misar_imputation/" #path to results
    os.makedirs(out_dir, exist_ok=True)
    ad_pred.write_h5ad(f'{out_dir}/cv{i}_imputedATAC.h5ad')

cv=1, missing=atac
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
	Completed 5 / 10 iteration(s).
	Completed 6 / 10 iteration(s).
	Completed 7 / 10 iteration(s).
	Completed 8 / 10 iteration(s).
	Completed 9 / 10 iteration(s).
	Completed 10 / 10 iteration(s).
Reach convergence after 10 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
Reach convergence after 2 iteration(s).
batch0: ['rna']
batch1: ['rna', 'atac']
batch2: ['rna', 'atac']
------Calculating spatial graph...
The graph contains 17770 edges, 1777 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 19490 edges, 1949 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 21290 edges, 2129 cells.
10.0000 neighbors per cell on average.
------Cal

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:02<00:00, 37.89it/s]


impute atac-counts for batch-1
==> cv 0, ATAC imputation: 
cv=2, missing=atac
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
	Completed 5 / 10 iteration(s).
	Completed 6 / 10 iteration(s).
	Completed 7 / 10 iteration(s).
	Completed 8 / 10 iteration(s).
	Completed 9 / 10 iteration(s).
	Completed 10 / 10 iteration(s).
Reach convergence after 10 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
Reach convergence after 2 iteration(s).
batch0: ['rna', 'atac']
batch1: ['rna']
batch2: ['rna', 'atac']
------Calculating spatial graph...
The graph contains 17770 edges, 1777 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 19490 edges, 1949 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 21290 edges, 2

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:02<00:00, 38.48it/s]


impute atac-counts for batch-2
==> cv 1, ATAC imputation: 
cv=3, missing=atac
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
	Completed 5 / 10 iteration(s).
	Completed 6 / 10 iteration(s).
	Completed 7 / 10 iteration(s).
	Completed 8 / 10 iteration(s).
	Completed 9 / 10 iteration(s).
	Completed 10 / 10 iteration(s).
Reach convergence after 10 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
Reach convergence after 2 iteration(s).
batch0: ['rna', 'atac']
batch1: ['rna', 'atac']
batch2: ['rna']
------Calculating spatial graph...
The graph contains 17770 edges, 1777 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 19490 edges, 1949 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 21290 edges, 2

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:02<00:00, 39.67it/s]


impute atac-counts for batch-3
==> cv 2, ATAC imputation: 
